# 01 — LangChain Fundamentals
### Models · Prompts · Output Parsers · Chains · Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1Dr4RO8capBWqoPS3DiQpW_tKlAjxXqFd#scrollTo=Jq5TQHZyj7D6)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://python.org)
[![LangChain](https://img.shields.io/badge/LangChain-0.3.x-green)](https://python.langchain.com)
[![Gemini](https://img.shields.io/badge/LLM-Gemini%201.5%20Flash-orange)](https://aistudio.google.com)


### Copy the links from below badge to visit my pages


[![GitHub](https://img.shields.io/badge/GitHub-View_Profile-black?logo=github)](https://github.com/mtptisid)
<a href="https://www.linkedin.com/in/siddharamayya-mathapati" target="_blank">
  <img src="https://img.shields.io/badge/LinkedIn-Connect-blue?logo=linkedin" />
</a>
[![Portfolio](https://img.shields.io/badge/Portfolio-Visit-orange?logo=google-chrome)](https://siddharamayya.in)


> **Part 1 of the LangChain Tutorial Series.**
> The complete beginner foundation — from your first LLM call to multi-turn conversations.
> No ML experience needed. No GPU needed. Just a free Gemini API key.

---

## What You Will Build

By the end of this notebook you will have:

- Made your first LLM call using Gemini via LangChain
- Built reusable prompt templates with dynamic variables
- Parsed LLM output into plain strings and structured JSON
- Chained multiple steps together using the `|` pipe operator (LCEL)
- Built a chatbot that remembers previous messages across turns

---

## Prerequisites

**Python knowledge:** Basic — variables, functions, loops, imports.
**AI/ML knowledge:** None required.
**API key:** Free Gemini key from [aistudio.google.com](https://aistudio.google.com) — no credit card needed.

---

## Setup

### Install dependencies

In [1]:
!pip install -q -U langchain langchain-community langchain-google-genai \
            langchain-text-splitters google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


### Set your API key

In Google Colab, store your key in **Secrets** (the 🔑 icon in the left sidebar)
with the name `GOOGLE_API_KEY`. Then load it:

In [2]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

> Never paste your API key directly in a notebook cell — it will be visible
> to anyone you share the notebook with.

---

## Part 1 — Your First LLM Call

The simplest possible LangChain program: ask a question, get an answer.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

response = llm.invoke([HumanMessage(content="What is the capital of France?")])
print(response.content)
# → "The capital of France is Paris."

The capital of France is **Paris**.


**Key concepts:**
- `ChatGoogleGenerativeAI` wraps the Gemini API — handles auth, retries, response parsing
- `HumanMessage` represents a message from the user. LangChain also has `SystemMessage` and `AIMessage`
- `temperature=0` makes output deterministic — same question always gives same answer
- `.invoke()` runs the chain and waits for the full response

**Swap LLMs in one line — everything else stays identical:**

```python
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

from langchain_community.llms import Ollama
llm = Ollama(model="llama3")   # fully local, no API key
```


## Part 2 — Prompt Templates

Hardcoding questions doesn't scale. Prompt templates let you define
a reusable structure and fill in variables at runtime.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant who explains things simply."),
    ("human", "Explain {topic} in 3 bullet points for a beginner.")
])

chain = prompt | llm
result = chain.invoke({"topic": "machine learning"})
print(result.content)

Here's machine learning in a nutshell:

*   **It's like teaching a computer by showing it lots of examples,** instead of writing out every single rule for it to follow.
*   **The computer then learns to spot patterns and make connections** in that data all by itself.
*   **Once it learns, it can use those patterns to make smart predictions or decisions** on new information, and often gets better over time.


**Why templates over f-strings?**

| f-string | ChatPromptTemplate |
|---|---|
| Just a string | Typed — system, human, ai roles |
| No reuse structure | Composable and serializable |
| Manual formatting | Automatic message structuring |

**Multiple variables:**

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} who responds in {language}."),
    ("human", "{question}")
])

result = (prompt | llm).invoke({
    "role": "doctor",
    "language": "simple English",
    "question": "What causes high blood pressure?"
})


## Part 3 — Output Parsers

By default `.invoke()` returns an `AIMessage` object. Output parsers
transform that into something directly usable in your code.

In [7]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "neural networks", "role": "engineer",
                        "language": "simple English",
                        "question": "neural networks"
                        })

print(type(result))   # <class 'str'>  ← plain string, not AIMessage
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
Okay, neural networks.

They are like computer brains.

They learn from examples, just like people do.

We use them to help computers recognize things, make decisions, or understand language.



**JSON output — get structured data back from the LLM:**

In [9]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class MovieReview(BaseModel):
    title:   str   = Field(description="movie title")
    rating:  float = Field(description="rating out of 10")
    summary: str   = Field(description="one sentence summary")

parser = JsonOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie critic. Respond with valid JSON only."),
    ("human", "Review the movie: {movie}\n\n{format_instructions}")
]).partial(format_instructions=parser.get_format_instructions())

chain  = prompt | llm | parser
result = chain.invoke({"movie": "Inception"})

print(result["title"])    # Inception
print(result["rating"])   # 9.2

Inception
9.5



## Part 4 — Chains (LCEL)

**LCEL** (LangChain Expression Language) connects steps with `|`.
The output of each step flows into the next — like Unix pipes.

```
prompt | llm | output_parser
  ↓         ↓              ↓
template  AIMessage      str / dict
```

**Sequential chain — output of step 1 feeds into step 2:**

In [15]:
from langchain_core.output_parsers import StrOutputParser

outline_prompt = ChatPromptTemplate.from_template(
    "Create a 5-point blog post outline about: {topic}"
)
write_prompt = ChatPromptTemplate.from_template(
    "Write a blog post based on this outline:\n{outline}"
)

outline_chain = outline_prompt | llm | StrOutputParser()
write_chain   = write_prompt   | llm | StrOutputParser()

full_chain = outline_chain | (lambda outline: {"outline": outline}) | write_chain
result = full_chain.invoke({"topic": "the future of AI"})
print(result)

## Unlocking Tomorrow: A Glimpse into the Future of AI

From the smart assistant that wakes you up to the personalized recommendations shaping your entertainment, Artificial Intelligence is no longer a futuristic concept – it's woven into the very fabric of our daily lives. The rapid advancements in machine learning, deep learning, and natural language processing are pushing the boundaries of what we thought possible, transforming industries and redefining human-computer interaction at an astonishing pace. This isn't just a technological shift; it's a societal evolution. This post will explore the exciting, transformative, and challenging trajectory of AI, looking beyond the present to what truly lies ahead. By the end, you'll have a clearer understanding of AI's immense potential impact on your life and the world around us.

---

### 1. The Foundation: Where We Are & What's Driving It

Today's AI landscape is nothing short of revolutionary. We've witnessed the rise of sophisticated la

**Parallel chain — run two LLM calls simultaneously:**

In [11]:
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    pros=(ChatPromptTemplate.from_template("List 3 pros of {tech}") | llm | StrOutputParser()),
    cons=(ChatPromptTemplate.from_template("List 3 cons of {tech}") | llm | StrOutputParser()),
)

result = parallel.invoke({"tech": "React"})
print(result["pros"])
print(result["cons"])

Here are 3 pros of React:

1.  **Efficient Updates with Virtual DOM:** React uses a **Virtual DOM** to optimize updates to the actual DOM. Instead of re-rendering the entire page on every change, it calculates the minimal changes needed and updates only those specific parts. This leads to faster performance, a smoother user experience, and less strain on the browser.

2.  **Component-Based Architecture for Reusability and Maintainability:** React's core is its **component-based structure**, which encourages breaking down the UI into small, independent, and reusable pieces. This modularity makes development faster, code easier to manage, test, and scale for larger applications, as components can be developed and maintained in isolation.

3.  **Large Community and Rich Ecosystem:** React boasts a **massive and active community**, meaning abundant resources, tutorials, and readily available solutions to common problems. Its rich ecosystem includes a vast array of libraries, tools, and fra

**Batch — process multiple inputs efficiently:**

In [17]:
prompt = ChatPromptTemplate.from_template("explain {topic} in few words")
batch_chain = prompt | llm |  StrOutputParser()

# All 3 run in parallel — much faster than 3 separate .invoke() calls
results = batch_chain.batch([
    {"topic": "neural networks"},
    {"topic": "transformers"},
    {"topic": "reinforcement learning"},
])

print(results)

['**Brain-inspired computer systems that learn from data to recognize patterns and make predictions.**', 'A neural network architecture that processes sequences (like text) by using **attention** to understand the relationships and context between all parts of the sequence simultaneously.', '**Trial and error learning.** An agent takes actions in an environment, receives rewards (or penalties), and learns to choose actions that maximize its cumulative reward over time.']



## Part 5 — Memory (Multi-Turn Conversations)

LLMs have no memory between calls — each call is independent.
You must track and pass conversation history manually.
LangChain's `RunnableWithMessageHistory` handles this automatically.

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),   # ← past messages injected here
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

# Session store — in production, replace with Redis or a database
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_001"}}

r1 = chain_with_memory.invoke({"input": "My name is Arjun."}, config=config)
print(r1)   # "Nice to meet you, Arjun!"

r2 = chain_with_memory.invoke({"input": "What is my name?"}, config=config)
print(r2)   # "Your name is Arjun."  ← it remembers!

Nice to meet you, Arjun!
Your name is Arjun.


**How it works:**

```
Turn 1: [HumanMessage("My name is Arjun.")]  →  LLM  →  "Nice to meet you!"
                                                            ↓ stored in session
Turn 2: [HumanMessage("My name is Arjun."),             ← history injected
         AIMessage("Nice to meet you!"),
         HumanMessage("What is my name?")]   →  LLM  →  "Your name is Arjun."
```

---

## Common Errors in This Notebook

| Error | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: langchain.text_splitter` | Moved to separate package | `pip install langchain-text-splitters` |
| `ModuleNotFoundError: langchain.embeddings` | Moved to community | `pip install langchain-community` |
| `AuthenticationError` | API key not set or wrong | Check `os.environ["GOOGLE_API_KEY"]` |
| `ValueError: Missing input keys` | Variable name in prompt doesn't match `.invoke()` dict | Check `{variable}` names match exactly |

---

## Quick Reference


In [20]:
# 1. LLM
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 2. Prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Explain {topic} simply.")

# 3. Chain
from langchain_core.output_parsers import StrOutputParser
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "RAG"})

# 4. Parallel
from langchain_core.runnables import RunnableParallel
parallel = RunnableParallel(a=chain, b=batch_chain)

# 5. Memory
from langchain_core.runnables.history import RunnableWithMessageHistory
chain_with_memory = RunnableWithMessageHistory(chain, get_session_history,)


## What's Next

**[→ Notebook 02: RAG Pipeline](./02_rag_pipeline.ipynb)**
Load PDFs, chunk them, embed with HuggingFace, store in FAISS,
and build a full retrieval-augmented generation chain.

---

*Part of the [LangChain Tutorial Series](../README.md) — built with LangChain 0.3.x and Google Gemini 1.5 Flash*